In [2]:
import sys
print(sys.executable)

c:\Users\wilso\Downloads\Complete-Langchain-Tutorials-main\.venv\Scripts\python.exe


In [1]:
import os

from dotenv import load_dotenv
from pypdf import PdfReader

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_cohere import CohereEmbeddings, ChatCohere
from langchain_community.vectorstores import FAISS

load_dotenv()

COHERE_API_KEY = os.getenv("COHERE_API_KEY")

if not COHERE_API_KEY:
    raise ValueError("COHERE_API_KEY is missing from .env")

print("Cohere API key loaded successfully.")

Cohere API key loaded successfully.


C:\Users\wilso\AppData\Local\Temp\ipykernel_23840\334658408.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [2]:
def read_pdf(file_path):

    reader = PdfReader(file_path)

    text = ""

    for page in reader.pages:

        page_text = page.extract_text()

        if page_text:
            text += page_text + "\n"

    return text


pdf_path = "documents/budget_speech.pdf"

text = read_pdf(pdf_path)

print("Characters extracted:", len(text))

Characters extracted: 89331


In [3]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=50
)

chunks = text_splitter.split_text(text)

print("Number of chunks:", len(chunks))

Number of chunks: 118


In [4]:
embeddings = CohereEmbeddings(
    model="embed-v4.0",
    cohere_api_key=COHERE_API_KEY
)

print("Embedding model loaded successfully.")

Embedding model loaded successfully.


In [5]:
vector_store = FAISS.from_texts(
    chunks,
    embedding=embeddings
)

print("FAISS vector database created successfully.")

FAISS vector database created successfully.


In [6]:
llm = ChatCohere(
    model="command-a-03-2025",
    temperature=0.3,
    cohere_api_key=COHERE_API_KEY
)

print("Cohere chat model loaded successfully.")

Cohere chat model loaded successfully.


In [7]:
def retrieve_answers(query, k=3):

    docs = vector_store.similarity_search(
        query,
        k=k
    )

    context = "\n\n".join(
        doc.page_content
        for doc in docs
    )

    prompt = f"""
You are a question-answering assistant.

Answer the question using ONLY the context provided below.

If the answer is not available in the context, say:

"Answer is not available in the document."

Do not make up information.

CONTEXT:
{context}

QUESTION:
{query}

ANSWER:
"""

    response = llm.invoke(prompt)

    return response.content

In [8]:
our_query = "How much will the agriculture credit target be increased to?"

answer = retrieve_answers(our_query)

print(answer)

The agriculture credit target will be increased to ` 20 lakh crore.
